# Trabalho 02 — Deep Learning (PPGCA/Unisinos, 2026/2)
**Detecção de *prompt injection* com um MLP sobre TF-IDF: do backpropagation à mão aos otimizadores**

Aluno: Epitácio Neto · Professor: Felipe André Zeiser

**Problema.** Aplicações com modelos de linguagem recebem texto não confiável (o *prompt* do usuário). Um *prompt injection*
tenta fazer o modelo ignorar as instruções originais (p.ex. "ignore as instruções anteriores e..."). O objetivo é um detector
**barato e rápido** que classifique cada texto como *seguro* (0) ou *injeção* (1) e funcione como primeiro filtro.

**Escopo e relação com as aulas.** O trabalho aplica, em uma tarefa real, o que foi visto: texto → tokens → TF-IDF (Aula 6);
rede densa (MLP) com ReLU/sigmoide e perda de entropia cruzada binária (Aulas 1 e 2); backpropagation à mão e verificação
numérica (Aulas 2 e 3); comparação dos otimizadores SGD, Momentum, RMSProp e Adam (Aula 4); escolha de arquitetura justificada
frente a um modelo pré-treinado (Aulas 3 e 5). Apenas **datasets públicos** são usados.

**Como executar.** `python trabalho2_dl.py` (CPU, ~10 min) ou abrir o notebook. Os datasets são baixados do Hugging Face Hub
(`deepset/prompt-injections`, `neuralchemy/Prompt-injection-dataset` config `core`, `xTRam1/safe-guard-prompt-injection`).
Resultados em `resultados/` e figuras em `figuras/`. Um laboratório web interativo, que repete os experimentos no navegador,
acompanha o trabalho (ver README do repositório).

**Protocolo.** Treino/validação fixos (80/20 do treino do deepset, estratificado); **todas as escolhas** (taxa de aprendizado,
tamanho da camada oculta, época de parada) usam só a validação; o teste do deepset (em-distribuição) e dois conjuntos
fora-da-distribuição (neuralchemy, xTRam1) são apenas relatados. 5 sementes (0 a 4); média ± desvio-padrão.

### 0. Preparação do ambiente
A primeira célula de código instala as dependências no Colab (no-op se já instaladas) e a seguinte grava o módulo auxiliar `injecao.py` (tokenização, TF-IDF à mão, MLP com backpropagation à mão e treino PyTorch), para que o notebook seja autossuficiente.

In [1]:
import importlib.util, subprocess, sys
faltam = [p for p, m in (('datasets', 'datasets'), ('scikit-learn', 'sklearn'), ('scipy', 'scipy'), ('torch', 'torch')) if importlib.util.find_spec(m) is None]
if faltam:
    subprocess.check_call([sys.executable, '-m', 'pip', '-q', 'install', *faltam])
print('dependências OK; faltavam:', faltam)

dependências OK; faltavam: []


In [2]:
%%writefile injecao.py
"""Biblioteca do Trabalho 02 (Deep Learning, PPGCA/Unisinos 2026/2): detecção de prompt injection com MLP sobre TF-IDF.

Contém (i) carga e divisão dos dados, (ii) tokenização e TF-IDF implementados à mão (para que o laboratório web em
JavaScript reproduza exatamente os mesmos números), (iii) um MLP em NumPy com backpropagation escrito à mão (Aulas 2 e 3),
(iv) o mesmo MLP em PyTorch com os quatro otimizadores da Aula 4 e (v) métricas.
"""
import math
import os
import re
import time

import numpy as np

# ----------------------------------------------------------------------------------------------------------------------
# 1. Dados
# ----------------------------------------------------------------------------------------------------------------------
SEMENTE_DIVISAO = 2026


def carregar_dados(n_xtram=1000):
    """Baixa os datasets do Hugging Face Hub (ou usa o cache local; defina HF_HUB_OFFLINE=1 para forçar o modo offline).
    deepset (treino/teste) = em-distribuição; neuralchemy (teste) e xTRam1 (amostra do teste) = fora da distribuição."""
    from datasets import load_dataset
    ds = load_dataset("deepset/prompt-injections")
    nc = load_dataset("neuralchemy/Prompt-injection-dataset", "core")["test"]
    xt = load_dataset("xTRam1/safe-guard-prompt-injection")["test"]
    rng = np.random.RandomState(SEMENTE_DIVISAO)
    # amostra estratificada e determinística do xTRam1 (mesma regra do Trabalho 01: n=1000)
    y = np.array(xt["label"])
    idx = []
    for c in (0, 1):
        ic = np.where(y == c)[0]
        k = int(round(n_xtram * len(ic) / len(y)))
        idx += list(rng.choice(ic, k, replace=False))
    idx = sorted(idx)
    d = {
        "treino_total": (list(ds["train"]["text"]), np.array(ds["train"]["label"])),
        "teste": (list(ds["test"]["text"]), np.array(ds["test"]["label"])),
        "neuralchemy": (list(nc["text"]), np.array(nc["label"])),
        "xtram1": ([xt["text"][i] for i in idx], y[idx]),
    }
    return d


def dividir_treino_val(textos, y, frac_val=0.2, semente=SEMENTE_DIVISAO):
    """Divisão estratificada fixa treino/validação (a validação serve a TODAS as escolhas; o teste só ao final)."""
    rng = np.random.RandomState(semente)
    iv = []
    for c in (0, 1):
        ic = np.where(y == c)[0]
        iv += list(rng.choice(ic, int(round(frac_val * len(ic))), replace=False))
    iv = np.array(sorted(iv))
    it = np.array([i for i in range(len(y)) if i not in set(iv.tolist())])
    return ([textos[i] for i in it], y[it]), ([textos[i] for i in iv], y[iv])


# ----------------------------------------------------------------------------------------------------------------------
# 2. Tokenização e TF-IDF (à mão; o JavaScript do laboratório faz exatamente o mesmo)
# ----------------------------------------------------------------------------------------------------------------------
_RE_TOKEN = re.compile(r"\w+")


def tokenizar(texto, bigramas=False):
    t = _RE_TOKEN.findall(texto.lower())
    if bigramas:
        t = t + [t[i] + " " + t[i + 1] for i in range(len(t) - 1)]
    return t


class Vetorizador:
    """modo: 'tfidf' (padrão), 'bow' (contagem binária). idf = ln((1+N)/(1+df)) + 1; linhas normalizadas em L2."""

    def __init__(self, max_vocab=2000, min_df=2, bigramas=False, modo="tfidf"):
        self.max_vocab, self.min_df, self.bigramas, self.modo = max_vocab, min_df, bigramas, modo

    def ajustar(self, textos):
        df = {}
        for t in textos:
            for w in set(tokenizar(t, self.bigramas)):
                df[w] = df.get(w, 0) + 1
        itens = [(w, c) for w, c in df.items() if c >= self.min_df]
        itens.sort(key=lambda x: (-x[1], x[0]))           # desempate alfabético (determinístico)
        itens = itens[: self.max_vocab]
        self.vocab = [w for w, _ in itens]
        self.indice = {w: i for i, w in enumerate(self.vocab)}
        n = len(textos)
        self.idf = np.array([math.log((1 + n) / (1 + c)) + 1 for _, c in itens], dtype=np.float64)
        return self

    def transformar(self, textos):
        X = np.zeros((len(textos), len(self.vocab)), dtype=np.float64)
        for r, t in enumerate(textos):
            for w in tokenizar(t, self.bigramas):
                j = self.indice.get(w)
                if j is not None:
                    X[r, j] += 1.0
        if self.modo == "bow":
            X = (X > 0).astype(np.float64)
        else:
            X = X * self.idf
        nrm = np.linalg.norm(X, axis=1, keepdims=True)
        nrm[nrm == 0] = 1.0
        return X / nrm


# ----------------------------------------------------------------------------------------------------------------------
# 3. MLP em NumPy com backpropagation escrito à mão (Aulas 2 e 3)
# ----------------------------------------------------------------------------------------------------------------------
def sigmoide(z):
    return 1.0 / (1.0 + np.exp(-z))


def bce(p, y, eps=1e-12):
    p = np.clip(p, eps, 1 - eps)
    return float(-np.mean(y * np.log(p) + (1 - y) * np.log(1 - p)))


def mlp_forward(P, X):
    """X(n,V) -> z1=X@W1+b1 -> h=ReLU(z1) -> z2=h@W2+b2 -> p=sigmoide(z2)."""
    z1 = X @ P["W1"] + P["b1"]
    h = np.maximum(z1, 0.0)
    z2 = h @ P["W2"] + P["b2"]
    return z1, h, z2, sigmoide(z2)


def mlp_gradientes(P, X, y):
    """Regra da cadeia à mão. Com BCE+sigmoide, dL/dz2 = (p - y)/n (Aula 2)."""
    n = X.shape[0]
    z1, h, z2, p = mlp_forward(P, X)
    dz2 = (p - y.reshape(-1, 1)) / n
    g = {"W2": h.T @ dz2, "b2": dz2.sum(0)}
    dh = dz2 @ P["W2"].T
    dz1 = dh * (z1 > 0)
    g["W1"] = X.T @ dz1
    g["b1"] = dz1.sum(0)
    return g, bce(p.ravel(), y)


def n_parametros(V, H):
    """V*H + H (camada oculta) + H*1 + 1 (saída). Mesmo raciocínio dos 641 parâmetros da Aula 4."""
    return V * H + H + H + 1


# ----------------------------------------------------------------------------------------------------------------------
# 4. MLP em PyTorch com os otimizadores da Aula 4
# ----------------------------------------------------------------------------------------------------------------------
OTIMIZADORES = {
    "SGD": lambda ps, lr: __import__("torch").optim.SGD(ps, lr=lr),
    "SGD+Momentum": lambda ps, lr: __import__("torch").optim.SGD(ps, lr=lr, momentum=0.9),
    "RMSProp": lambda ps, lr: __import__("torch").optim.RMSprop(ps, lr=lr, alpha=0.9, eps=1e-8),
    "Adam": lambda ps, lr: __import__("torch").optim.Adam(ps, lr=lr, betas=(0.9, 0.999), eps=1e-8),
}


def criar_mlp(V, H, semente):
    import torch
    torch.manual_seed(semente)
    if H == 0:   # regressão logística (sem camada oculta)
        return torch.nn.Sequential(torch.nn.Linear(V, 1))
    return torch.nn.Sequential(torch.nn.Linear(V, H), torch.nn.ReLU(), torch.nn.Linear(H, 1))


def metricas(y, p, limiar=0.5):
    from sklearn.metrics import roc_auc_score
    y = np.asarray(y)
    pred = (p >= limiar).astype(int)
    tp = int(((pred == 1) & (y == 1)).sum()); fp = int(((pred == 1) & (y == 0)).sum())
    fn = int(((pred == 0) & (y == 1)).sum()); tn = int(((pred == 0) & (y == 0)).sum())
    prec = tp / (tp + fp) if tp + fp else 0.0
    rec = tp / (tp + fn) if tp + fn else 0.0
    f1 = 2 * prec * rec / (prec + rec) if prec + rec else 0.0
    auc = float(roc_auc_score(y, p)) if len(set(y.tolist())) == 2 else float("nan")
    return {"acc": (tp + tn) / len(y), "prec": prec, "rec": rec, "f1": f1, "auc": auc, "fpr": fp / (fp + tn) if fp + tn else 0.0,
            "tp": tp, "fp": fp, "fn": fn, "tn": tn}


def treinar_mlp(Xtr, ytr, Xva, yva, H=32, otim="Adam", lr=1e-3, epocas=40, batch=32, semente=0, patience=None):
    """Treina com mini-batch e devolve o modelo na MELHOR época (menor perda de validação) e as curvas por época."""
    import copy
    import torch
    torch.set_num_threads(1)
    m = criar_mlp(Xtr.shape[1], H, semente)
    opt = OTIMIZADORES[otim](m.parameters(), lr)
    lossf = torch.nn.BCEWithLogitsLoss()
    Xt = torch.tensor(Xtr, dtype=torch.float32); yt = torch.tensor(ytr, dtype=torch.float32).view(-1, 1)
    Xv = torch.tensor(Xva, dtype=torch.float32); yv = torch.tensor(yva, dtype=torch.float32).view(-1, 1)
    g = torch.Generator().manual_seed(semente)
    curva = {"treino": [], "val": [], "val_acc": [], "val_f1": []}
    melhor, melhor_ep, estado = float("inf"), 0, copy.deepcopy(m.state_dict())
    for ep in range(1, epocas + 1):
        m.train()
        perm = torch.randperm(len(Xt), generator=g)
        for i in range(0, len(Xt), batch):
            b = perm[i:i + batch]
            opt.zero_grad()
            lossf(m(Xt[b]), yt[b]).backward()
            opt.step()
        m.eval()
        with torch.no_grad():
            lt = float(lossf(m(Xt), yt)); zv = m(Xv); lv = float(lossf(zv, yv))
            pv = torch.sigmoid(zv).numpy().ravel()
        mt = metricas(yva, pv)
        curva["treino"].append(lt); curva["val"].append(lv); curva["val_acc"].append(mt["acc"]); curva["val_f1"].append(mt["f1"])
        if lv < melhor - 1e-9:
            melhor, melhor_ep, estado = lv, ep, copy.deepcopy(m.state_dict())
        if patience and ep - melhor_ep >= patience:
            break
    m.load_state_dict(estado)
    m.eval()
    return m, curva, melhor_ep


def prever(m, X):
    import torch
    with torch.no_grad():
        return torch.sigmoid(m(torch.tensor(X, dtype=torch.float32))).numpy().ravel()


def ic95(vals):
    """Média, desvio-padrão (ddof=1) e IC95% t-Student entre sementes."""
    from scipy import stats
    v = np.asarray(vals, dtype=float)
    mu, sd = float(v.mean()), float(v.std(ddof=1)) if len(v) > 1 else 0.0
    h = float(stats.t.ppf(0.975, len(v) - 1) * sd / math.sqrt(len(v))) if len(v) > 1 else 0.0
    return mu, sd, mu - h, mu + h


Overwriting injecao.py


In [3]:
import json, os, sys, time, warnings
from pathlib import Path
import numpy as np, pandas as pd
warnings.filterwarnings("ignore")
AQUI = Path(__file__).resolve().parent if "__file__" in globals() else Path.cwd()
sys.path.insert(0, str(AQUI))
import injecao as I
RAIZ = AQUI.parent if (AQUI.parent / "resultados").exists() else AQUI
RES, FIG = RAIZ / "resultados", RAIZ / "figuras"
RES.mkdir(exist_ok=True); FIG.mkdir(exist_ok=True)
sys.path.insert(0, "/home/ubuntu/meridianseal/unisinos/tools/templates")
try:
    from plot_theme import aplicar, salvar, COR
    aplicar()
except Exception:
    import matplotlib; matplotlib.use("Agg")
    COR = {"azul": "#1D2369", "azul2": "#4A5AB8", "vermelho": "#C0392B", "cinza": "#666666", "claro": "#E9ECF3", "verde": "#2E7D5B"}
    def aplicar(): pass
    def salvar(fig, p): fig.tight_layout(); fig.savefig(p, dpi=150, bbox_inches="tight")
import matplotlib.pyplot as plt
import torch, sklearn
SEMENTES = [0, 1, 2, 3, 4]
EPOCAS, BATCH = 40, 32
INFO = {"python": sys.version.split()[0], "torch": torch.__version__, "sklearn": sklearn.__version__, "numpy": np.__version__,
        "epocas": EPOCAS, "batch": BATCH, "sementes": SEMENTES}
def salvar_json(obj, nome): (RES / nome).write_text(json.dumps(obj, ensure_ascii=False, indent=1))

## 1. Dados
O deepset (546 treino / 116 teste) é pequeno e **bilíngue** (inglês e alemão). Os conjuntos fora-da-distribuição servem para
medir quanto o detector depende do vocabulário visto no treino.

In [4]:
D = I.carregar_dados()
(tr_t, tr_y), (va_t, va_y) = I.dividir_treino_val(*D["treino_total"])
te_t, te_y = D["teste"]
CONJ = {"deepset (teste)": D["teste"], "neuralchemy (OOD)": D["neuralchemy"], "xTRam1 (OOD)": D["xtram1"]}
tab = [{"conjunto": "treino", "n": len(tr_y), "injecao": int(tr_y.sum())}, {"conjunto": "validação", "n": len(va_y), "injecao": int(va_y.sum())}]
tab += [{"conjunto": k, "n": len(v[1]), "injecao": int(v[1].sum())} for k, v in CONJ.items()]
tab_dados = pd.DataFrame(tab); tab_dados["pct_injecao"] = (100 * tab_dados.injecao / tab_dados.n).round(1)
print(tab_dados.to_string(index=False)); tab_dados.to_csv(RES / "tab_dados.csv", index=False)
print("Exemplos:", [(t[:70], int(l)) for t, l in list(zip(tr_t, tr_y))[:3]])

Using the latest cached version of the dataset since deepset/prompt-injections couldn't be found on the Hugging Face Hub (offline mode is enabled).


Found the latest cached dataset configuration 'default' at /home/ubuntu/.cache/huggingface/datasets/deepset___prompt-injections/default/0.0.0/4f61ecb038e9c3fb77e21034b22511b523772cdd (last modified on Thu Oct  8 18:23:12 2026).


Using the latest cached version of the dataset since neuralchemy/Prompt-injection-dataset couldn't be found on the Hugging Face Hub (offline mode is enabled).


Found the latest cached dataset configuration 'core' at /home/ubuntu/.cache/huggingface/datasets/neuralchemy___prompt-injection-dataset/core/0.0.0/7d70432dfcf47a821612cbf9d34e9d9e3ad20e75 (last modified on Thu Oct  8 18:23:33 2026).


Using the latest cached version of the dataset since xTRam1/safe-guard-prompt-injection couldn't be found on the Hugging Face Hub (offline mode is enabled).


Found the latest cached dataset configuration 'default' at /home/ubuntu/.cache/huggingface/datasets/xTRam1___safe-guard-prompt-injection/default/0.0.0/a3a877d608f37b7d20d9945671902df895ecdb46 (last modified on Thu Oct  8 18:23:21 2026).


         conjunto    n  injecao  pct_injecao
           treino  436      162         37.2
        validação  110       41         37.3
  deepset (teste)  116       60         51.7
neuralchemy (OOD)  942      552         58.6
     xTRam1 (OOD) 1000      316         31.6
Exemplos: [('Refugee crisis in Europe solutions', 0), ('My son is very interested in programming and would like to develop his', 0), ('I am looking for a new book and would like to know which current bests', 0)]


## 2. Texto → vetor (Aula 6)
Tokenização por palavras em minúsculas, vocabulário dos 2.000 termos mais frequentes (documento-frequência ≥ 2) e
**TF-IDF** com normalização L2, implementado à mão para que o laboratório em JavaScript produza os mesmos vetores.

In [5]:
V = I.Vetorizador(max_vocab=2000, min_df=2).ajustar(tr_t)
Xtr, Xva = V.transformar(tr_t), V.transformar(va_t)
XC = {k: V.transformar(v[0]) for k, v in CONJ.items()}
print("vocabulário:", len(V.vocab), "| tamanho do vetor:", Xtr.shape[1], "| 10 termos mais frequentes:", V.vocab[:10])
# verificação cruzada com o TfidfVectorizer do scikit-learn (mesma fórmula): diferença máxima deve ser ~0
from sklearn.feature_extraction.text import TfidfVectorizer
sk = TfidfVectorizer(tokenizer=I.tokenizar, lowercase=False, token_pattern=None, vocabulary=V.indice, norm="l2", smooth_idf=True)
sk.fit(tr_t)
dif_sk = float(np.abs(sk.transform(va_t).toarray() - Xva).max())
print("diferença máxima vs scikit-learn:", dif_sk); INFO["dif_tfidf_sklearn"] = dif_sk

vocabulário: 983 | tamanho do vetor: 983 | 10 termos mais frequentes: ['in', 'the', 'is', 'a', 'and', 'to', 'what', 'i', 'you', 'die']


diferença máxima vs scikit-learn: 1.1102230246251565e-16


## 3. Backpropagation à mão (Aulas 2 e 3)
Rede 2-camadas: `z1 = X·W1 + b1 → h = ReLU(z1) → z2 = h·W2 + b2 → p = σ(z2)`, perda BCE.
Com BCE + sigmoide, ∂L/∂z2 = (p − y)/n; a regra da cadeia propaga de trás para frente. A verificação compara os gradientes
escritos à mão com os do autograd do PyTorch (devem coincidir até a precisão numérica).

In [6]:
rng = np.random.RandomState(0)
H0 = 16
P = {"W1": rng.randn(Xtr.shape[1], H0) * np.sqrt(2 / Xtr.shape[1]), "b1": np.zeros(H0), "W2": rng.randn(H0, 1) * np.sqrt(2 / H0), "b2": np.zeros(1)}
xb, yb = Xtr[:64], tr_y[:64].astype(float)
g_mao, perda_mao = I.mlp_gradientes(P, xb, yb)
tp = {k: torch.tensor(v, dtype=torch.float64, requires_grad=True) for k, v in P.items()}
z = torch.relu(torch.tensor(xb) @ tp["W1"] + tp["b1"]) @ tp["W2"] + tp["b2"]
perda_t = torch.nn.functional.binary_cross_entropy_with_logits(z, torch.tensor(yb).view(-1, 1)); perda_t.backward()
dif_g = {k: float(np.abs(g_mao[k] - tp[k].grad.numpy()).max()) for k in P}
print("perda (mão):", round(perda_mao, 8), "| perda (torch):", round(float(perda_t), 8)); print("erro máx. dos gradientes:", dif_g)
INFO["gradcheck_max_abs_err"] = max(dif_g.values())
# um passo de gradiente descendente à mão reduz a perda (Aula 3)?
P2 = {k: P[k] - 0.5 * g_mao[k] for k in P}
print("perda antes/depois de 1 passo (lr=0,5):", round(perda_mao, 5), "→", round(I.mlp_gradientes(P2, xb, yb)[1], 5))

perda (mão): 0.69275854 | perda (torch): 0.69275854
erro máx. dos gradientes: {'W1': 1.734723475976807e-18, 'b1': 2.7755575615628914e-17, 'W2': 3.469446951953614e-18, 'b2': 1.1102230246251565e-16}
perda antes/depois de 1 passo (lr=0,5): 0.69276 → 0.60731


## 4. Linha de base e escolha da taxa de aprendizado por otimizador (Aula 4)
Linha de base: regressão logística sobre o mesmo TF-IDF (Aula 6: baselines simples seguem válidos). Para cada otimizador,
uma grade de taxas de aprendizado, 3 sementes, **escolhida pela perda de validação**.

In [7]:
from sklearn.linear_model import LogisticRegression
base_val = {}
for C in (0.1, 1, 10, 100):
    lr_ = LogisticRegression(C=C, max_iter=2000).fit(Xtr, tr_y)
    p = lr_.predict_proba(Xva)[:, 1]; base_val[C] = I.bce(p, va_y)
C_best = min(base_val, key=base_val.get); print("LogReg: perda de validação por C:", {k: round(v, 4) for k, v in base_val.items()}, "→ C =", C_best)
LR_GRID = {"SGD": [0.01, 0.1, 1.0, 3.0, 10.0], "SGD+Momentum": [0.001, 0.01, 0.1, 0.3, 1.0],
           "RMSProp": [1e-4, 1e-3, 1e-2, 1e-1], "Adam": [1e-4, 1e-3, 1e-2, 1e-1]}
H_PADRAO = 32
linhas = []
t0 = time.time()
for otim, grade in LR_GRID.items():
    for lr in grade:
        for s in SEMENTES[:3]:
            m, cv, ep = I.treinar_mlp(Xtr, tr_y, Xva, va_y, H=H_PADRAO, otim=otim, lr=lr, epocas=EPOCAS, batch=BATCH, semente=s)
            linhas.append({"otimizador": otim, "lr": lr, "semente": s, "val_loss": min(cv["val"]), "melhor_epoca": ep})
grade_df = pd.DataFrame(linhas)
resumo_lr = grade_df.groupby(["otimizador", "lr"]).agg(val_loss=("val_loss", "mean"), melhor_epoca=("melhor_epoca", "mean")).reset_index()
resumo_lr.to_csv(RES / "tab_grade_lr.csv", index=False)
LR_BEST = {o: float(g.loc[g.val_loss.idxmin(), "lr"]) for o, g in resumo_lr.groupby("otimizador")}
print(resumo_lr.round(4).to_string(index=False)); print("melhores taxas (validação):", LR_BEST, f"[{time.time()-t0:.0f}s]")

LogReg: perda de validação por C: {0.1: 0.6127, 1: 0.4498, 10: 0.2698, 100: 0.2125} → C = 100


  otimizador      lr  val_loss  melhor_epoca
        Adam  0.0001    0.6044       40.0000
        Adam  0.0010    0.2389       40.0000
        Adam  0.0100    0.2357        5.3333
        Adam  0.1000    0.3133        2.0000
     RMSProp  0.0001    0.6296       40.0000
     RMSProp  0.0010    0.2547       38.3333
     RMSProp  0.0100    0.2347        5.3333
     RMSProp  0.1000    0.2905        2.0000
         SGD  0.0100    0.6588       40.0000
         SGD  0.1000    0.5252       40.0000
         SGD  1.0000    0.2033       23.0000
         SGD  3.0000    0.3557       30.3333
         SGD 10.0000    0.6607       11.0000
SGD+Momentum  0.0010    0.6588       40.0000
SGD+Momentum  0.0100    0.5415       40.0000
SGD+Momentum  0.1000    0.2074       23.3333
SGD+Momentum  0.3000    0.2130        9.3333
SGD+Momentum  1.0000    0.2552        8.6667
melhores taxas (validação): {'Adam': 0.01, 'RMSProp': 0.01, 'SGD': 1.0, 'SGD+Momentum': 0.1} [45s]


## 5. Experimento principal: SGD × Momentum × RMSProp × Adam (5 sementes)
**Hipótese (fixada antes dos resultados, Aula 4):** com a taxa certa para cada um, os métodos adaptativos (RMSProp/Adam)
atingem menor perda de validação em menos épocas do que o SGD puro; e a taxa ótima do SGD é uma ordem de grandeza (ou mais)
maior que a do Adam.

In [8]:
def metr_conj(model, limiar=0.5):
    return {k: I.metricas(CONJ[k][1], I.prever(model, XC[k]), limiar) for k in CONJ}
curvas, rows = {}, []
for otim in LR_GRID:
    cs = []
    for s in SEMENTES:
        m, cv, ep = I.treinar_mlp(Xtr, tr_y, Xva, va_y, H=H_PADRAO, otim=otim, lr=LR_BEST[otim], epocas=EPOCAS, batch=BATCH, semente=s)
        cs.append(cv)
        mm = metr_conj(m)
        for k, v in mm.items():
            rows.append({"otimizador": otim, "semente": s, "conjunto": k, "melhor_epoca": ep, "val_loss": min(cv["val"]), **v})
    curvas[otim] = {k: np.array([c[k] for c in cs]).tolist() for k in cs[0]}
principal = pd.DataFrame(rows); principal.to_csv(RES / "principal_otimizadores.csv", index=False)
salvar_json(curvas, "curvas_otimizadores.json")
def agg(df, cols, by):
    out = []
    for key, g in df.groupby(by, sort=False):
        r = dict(zip(by, key if isinstance(key, tuple) else (key,)))
        for c in cols:
            mu, sd, lo, hi = I.ic95(g[c]); r[c], r[c + "_dp"], r[c + "_lo"], r[c + "_hi"] = mu, sd, lo, hi
        out.append(r)
    return pd.DataFrame(out)
t_otim = agg(principal, ["f1", "auc", "acc", "prec", "rec", "val_loss", "melhor_epoca"], ["otimizador", "conjunto"])
t_otim["lr"] = t_otim.otimizador.map(LR_BEST); t_otim.to_csv(RES / "tab_otimizadores.csv", index=False)
print(t_otim[t_otim.conjunto == "deepset (teste)"][["otimizador", "lr", "val_loss", "melhor_epoca", "f1", "f1_dp", "auc", "auc_dp"]].round(4).to_string(index=False))
# perda de validação em épocas marcadas (velocidade de convergência)
marc = [1, 3, 5, 10, 20, 40]
conv = pd.DataFrame([{"otimizador": o, **{f"ep{e}": float(np.mean(np.array(c["val"])[:, e - 1])) for e in marc}} for o, c in curvas.items()])
conv.to_csv(RES / "tab_convergencia.csv", index=False); print(conv.round(4).to_string(index=False))

fig, ax = plt.subplots(1, 2, figsize=(9.2, 3.5))
cores = {"SGD": COR["cinza"], "SGD+Momentum": COR["azul2"], "RMSProp": COR["verde"], "Adam": COR["vermelho"]}
for o, c in curvas.items():
    for a, k, tt in ((ax[0], "treino", "Perda de treino"), (ax[1], "val", "Perda de validação")):
        arr = np.array(c[k]); mu, sd = arr.mean(0), arr.std(0, ddof=1); x = np.arange(1, len(mu) + 1)
        a.plot(x, mu, color=cores[o], label=f"{o} (lr={LR_BEST[o]:g})"); a.fill_between(x, mu - sd, mu + sd, color=cores[o], alpha=0.15); a.set_title(tt); a.set_xlabel("época")
ax[0].set_ylabel("entropia cruzada binária"); ax[1].legend(fontsize=7.5)
salvar(fig, FIG / "fig1_otimizadores.png")

findfont: Font family 'Arial' not found.


findfont: Font family 'Arial' not found.


findfont: Font family 'Arial' not found.


findfont: Font family 'Arial' not found.


findfont: Font family 'Arial' not found.


findfont: Font family 'Arial' not found.


... (592 outputs left out)

findfont: Font family 'Arial' not found.


findfont: Font family 'Arial' not found.


findfont: Font family 'Arial' not found.


findfont: Font family 'Arial' not found.


## 6. Capacidade da rede: tamanho da camada oculta (subajuste × sobreajuste)
Adam com a melhor taxa; H ∈ {0 (regressão logística), 2, 8, 32, 128, 512}. Contagem de parâmetros: V·H + H + H + 1.

In [9]:
rows = []
for H in (0, 2, 8, 32, 128, 512):
    for s in SEMENTES:
        m, cv, ep = I.treinar_mlp(Xtr, tr_y, Xva, va_y, H=H, otim="Adam", lr=LR_BEST["Adam"], epocas=EPOCAS, batch=BATCH, semente=s)
        mm = metr_conj(m)
        rows.append({"H": H, "semente": s, "parametros": (Xtr.shape[1] + 1) if H == 0 else I.n_parametros(Xtr.shape[1], H), "melhor_epoca": ep,
                     "treino_loss": cv["treino"][ep - 1], "val_loss": cv["val"][ep - 1], "f1_teste": mm["deepset (teste)"]["f1"],
                     "auc_teste": mm["deepset (teste)"]["auc"], "f1_nc": mm["neuralchemy (OOD)"]["f1"], "f1_xt": mm["xTRam1 (OOD)"]["f1"]})
cap = pd.DataFrame(rows); cap.to_csv(RES / "capacidade_runs.csv", index=False)
t_cap = agg(cap, ["treino_loss", "val_loss", "f1_teste", "auc_teste", "f1_nc", "f1_xt", "melhor_epoca"], ["H"]); t_cap["parametros"] = cap.groupby("H", sort=False).parametros.first().values
t_cap.to_csv(RES / "tab_capacidade.csv", index=False)
print(t_cap[["H", "parametros", "treino_loss", "val_loss", "f1_teste", "f1_teste_dp", "auc_teste"]].round(4).to_string(index=False))
H_BEST = int(t_cap.loc[t_cap[t_cap.H > 0].val_loss.idxmin(), "H"]); print("H escolhido pela validação:", H_BEST)
fig, ax = plt.subplots(figsize=(5.2, 3.3))
xs = np.arange(len(t_cap)); ax.plot(xs, t_cap.treino_loss, "o-", color=COR["azul"], label="treino"); ax.plot(xs, t_cap.val_loss, "s-", color=COR["vermelho"], label="validação")
ax.set_xticks(xs); ax.set_xticklabels([("0 (logística)" if h == 0 else str(h)) for h in t_cap.H], fontsize=7.5); ax.set_xlabel("neurônios na camada oculta (H)"); ax.set_ylabel("perda na melhor época"); ax.legend(); ax.set_title("Capacidade × generalização")
salvar(fig, FIG / "fig2_capacidade.png")

findfont: Font family 'Arial' not found.


findfont: Font family 'Arial' not found.


findfont: Font family 'Arial' not found.


findfont: Font family 'Arial' not found.


findfont: Font family 'Arial' not found.


findfont: Font family 'Arial' not found.


... (260 outputs left out)

findfont: Font family 'Arial' not found.


findfont: Font family 'Arial' not found.


findfont: Font family 'Arial' not found.


findfont: Font family 'Arial' not found.


## 7. Representação do texto (Aula 6): BoW × TF-IDF × TF-IDF com bigramas

In [10]:
rows = []
for nome, kw in (("BoW binário", dict(modo="bow")), ("TF-IDF (unigramas)", dict()), ("TF-IDF (uni+bigramas)", dict(bigramas=True))):
    Vz = I.Vetorizador(max_vocab=2000, min_df=2, **kw).ajustar(tr_t)
    a, b = Vz.transformar(tr_t), Vz.transformar(va_t)
    XCz = {k: Vz.transformar(v[0]) for k, v in CONJ.items()}
    for s in SEMENTES:
        m, cv, ep = I.treinar_mlp(a, tr_y, b, va_y, H=H_BEST, otim="Adam", lr=LR_BEST["Adam"], epocas=EPOCAS, batch=BATCH, semente=s)
        mm = {k: I.metricas(CONJ[k][1], I.prever(m, XCz[k])) for k in CONJ}
        rows.append({"representacao": nome, "semente": s, "val_loss": min(cv["val"]), "f1_teste": mm["deepset (teste)"]["f1"], "auc_teste": mm["deepset (teste)"]["auc"],
                     "f1_nc": mm["neuralchemy (OOD)"]["f1"], "f1_xt": mm["xTRam1 (OOD)"]["f1"]})
rep = pd.DataFrame(rows); rep.to_csv(RES / "representacao_runs.csv", index=False)
t_rep = agg(rep, ["val_loss", "f1_teste", "auc_teste", "f1_nc", "f1_xt"], ["representacao"]); t_rep.to_csv(RES / "tab_representacao.csv", index=False)
print(t_rep[["representacao", "val_loss", "f1_teste", "f1_teste_dp", "auc_teste", "f1_nc", "f1_xt"]].round(4).to_string(index=False))

        representacao  val_loss  f1_teste  f1_teste_dp  auc_teste  f1_nc  f1_xt
          BoW binário    0.2052    0.8728       0.0210     0.9739 0.7158 0.5368
   TF-IDF (unigramas)    0.2315    0.8733       0.0182     0.9695 0.7115 0.5341
TF-IDF (uni+bigramas)    0.2158    0.8830       0.0117     0.9576 0.6775 0.5542


## 8. Modelo final, comparação e generalização fora da distribuição
Configuração final escolhida **apenas pela validação**: Adam, H e taxa acima. Comparação com a regressão logística e com os
números do Trabalho 01 (DistilBERT/BERT-base, mesmo teste do deepset, 3 sementes), lidos de `trabalho1/resultados/`.

In [11]:
rows, final_models = [], []
for s in SEMENTES:
    m, cv, ep = I.treinar_mlp(Xtr, tr_y, Xva, va_y, H=H_BEST, otim="Adam", lr=LR_BEST["Adam"], epocas=EPOCAS, batch=BATCH, semente=s)
    final_models.append(m)
    for k, v in metr_conj(m).items(): rows.append({"modelo": f"MLP (H={H_BEST}, Adam)", "semente": s, "conjunto": k, **v})
lr_ = LogisticRegression(C=C_best, max_iter=2000).fit(Xtr, tr_y)
for k in CONJ: rows.append({"modelo": "Regressão logística + TF-IDF", "semente": 0, "conjunto": k, **I.metricas(CONJ[k][1], lr_.predict_proba(XC[k])[:, 1])})
final = pd.DataFrame(rows); final.to_csv(RES / "final_runs.csv", index=False)
t_final = agg(final, ["f1", "auc", "acc", "prec", "rec", "fpr"], ["modelo", "conjunto"]); t_final.to_csv(RES / "tab_final.csv", index=False)
print(t_final[["modelo", "conjunto", "f1", "f1_dp", "auc", "prec", "rec", "fpr"]].round(3).to_string(index=False))
T1 = Path("/home/ubuntu/meridianseal/unisinos/dl/trabalho1/resultados/tab_e2_desempenho.csv")
ref_t1 = pd.read_csv(T1)
ref_t1 = ref_t1[ref_t1.detector.isin(["DistilBERT", "BERT-base", "Regra"])][["detector", "conjunto", "f1", "f1_dp", "auc", "n_sementes"]]
ref_t1.to_csv(RES / "ref_trabalho1.csv", index=False); print(ref_t1.round(2).to_string(index=False))
# matriz de confusão da semente 0 no teste
mc = metr_conj(final_models[0])["deepset (teste)"]; INFO["matriz_confusao_s0"] = {k: mc[k] for k in ("tp", "fp", "fn", "tn")}

                      modelo          conjunto    f1  f1_dp   auc  prec   rec   fpr
            MLP (H=32, Adam)   deepset (teste) 0.873  0.018 0.970 0.972 0.793 0.025
            MLP (H=32, Adam) neuralchemy (OOD) 0.712  0.009 0.695 0.690 0.736 0.470
            MLP (H=32, Adam)      xTRam1 (OOD) 0.534  0.003 0.653 0.374 0.932 0.719
Regressão logística + TF-IDF   deepset (teste) 0.849  0.000 0.971 0.978 0.750 0.018
Regressão logística + TF-IDF neuralchemy (OOD) 0.669  0.000 0.649 0.681 0.658 0.436
Regressão logística + TF-IDF      xTRam1 (OOD) 0.533  0.000 0.619 0.372 0.940 0.732
  detector                         conjunto    f1  f1_dp   auc  n_sementes
     Regra deepset (teste, em-distribuição) 58.82    NaN 70.83           1
     Regra          xTRam1 (OOD, sintético) 46.92    NaN 64.84           1
     Regra                neuralchemy (OOD) 34.38    NaN 60.16           1
DistilBERT deepset (teste, em-distribuição) 91.89   3.50 99.38           3
DistilBERT          xTRam1 (OOD, sint

## 9. Interpretação e custo
Importância de cada termo por **oclusão**: queda média da probabilidade de injeção quando o termo é removido do texto (textos
de injeção do treino+teste do deepset em que o termo aparece). E latência de inferência na CPU (tokenização + TF-IDF + MLP).

In [12]:
m0 = final_models[0]
def prob_texto(t): return float(I.prever(m0, V.transformar([t]))[0])
alvo = [t for t, y in zip(tr_t + te_t, list(tr_y) + list(te_y)) if y == 1]
imp = {}
for t in alvo:
    base = prob_texto(t); toks = list(dict.fromkeys(I.tokenizar(t)))
    for w in toks:
        if w not in V.indice: continue
        t2 = " ".join(x for x in I.tokenizar(t) if x != w)
        imp.setdefault(w, []).append(base - prob_texto(t2))
topo = sorted(((w, float(np.mean(v)), len(v)) for w, v in imp.items() if len(v) >= 5), key=lambda x: -x[1])[:12]
pd.DataFrame(topo, columns=["termo", "queda_media_prob", "n_textos"]).to_csv(RES / "tab_termos_importantes.csv", index=False)
print(topo)
amostra = (te_t * 10)[:500]
t0 = time.perf_counter()
for t in amostra: prob_texto(t)
lat_ms = (time.perf_counter() - t0) / len(amostra) * 1000
INFO["latencia_mlp_ms_por_texto"] = lat_ms; INFO["parametros_mlp"] = I.n_parametros(Xtr.shape[1], H_BEST); INFO["H_final"] = H_BEST; INFO["lr_melhor"] = LR_BEST
INFO["vocab"] = len(V.vocab); INFO["C_logreg"] = C_best
print(f"latência MLP: {lat_ms:.3f} ms/texto | parâmetros: {INFO['parametros_mlp']:,}")

[('digo', 0.18147713616490363, 5), ('context', 0.1422542283932368, 6), ('alles', 0.08792429566383361, 10), ('code', 0.07650844193994999, 8), ('n', 0.07419607539971669, 6), ('question', 0.05924542744954427, 18), ('y', 0.05776388645172119, 5), ('provide', 0.05692996978759766, 5), ('everything', 0.05623912811279297, 24), ('generate', 0.05544954538345337, 5), ('schreibe', 0.054132634943181816, 11), ('before', 0.05351503587527467, 14)]
latência MLP: 0.172 ms/texto | parâmetros: 31,521


In [13]:
# Figura 3: generalização (F1) por conjunto
fig, ax = plt.subplots(figsize=(6.4, 3.4))
nomes = ["MLP (H=%d, Adam)" % H_BEST, "Regressão logística + TF-IDF"]
conjs = list(CONJ)
w = 0.2
for i, nm in enumerate(nomes):
    vals = [t_final[(t_final.modelo == nm) & (t_final.conjunto == c)].f1.iloc[0] for c in conjs]
    dp = [t_final[(t_final.modelo == nm) & (t_final.conjunto == c)].f1_dp.iloc[0] for c in conjs]
    ax.bar(np.arange(3) + (i - 1.5) * w, vals, w, yerr=dp, color=[COR["azul"], COR["azul2"]][i], label=nm, capsize=2)
for i, (det, cor) in enumerate((("DistilBERT", COR["vermelho"]), ("BERT-base", COR["verde"]))):
    vals = []
    for c in ["deepset (teste, em-distribuição)", "neuralchemy (OOD)", "xTRam1 (OOD, sintético)"]:
        r = ref_t1[(ref_t1.detector == det) & (ref_t1.conjunto == c)]; vals.append(r.f1.iloc[0] / 100)
    ax.bar(np.arange(3) + (i + 0.5 - 0.0) * w, vals, w, color=cor, label=f"{det} (Trabalho 01)")
ax.set_xticks(range(3)); ax.set_xticklabels(["deepset (teste)", "neuralchemy (OOD)", "xTRam1 (OOD)"], fontsize=8); ax.set_ylabel("F1"); ax.set_ylim(0, 1.32); ax.legend(fontsize=7, ncol=2, loc="upper right")
ax.set_title("Generalização: F1 por conjunto")
salvar(fig, FIG / "fig3_generalizacao.png")
salvar_json(INFO, "info.json")
print("OK — resultados em", RES)

findfont: Font family 'Arial' not found.


findfont: Font family 'Arial' not found.


findfont: Font family 'Arial' not found.


findfont: Font family 'Arial' not found.


findfont: Font family 'Arial' not found.


findfont: Font family 'Arial' not found.


... (269 outputs left out)

findfont: Font family 'Arial' not found.


findfont: Font family 'Arial' not found.


findfont: Font family 'Arial' not found.


OK — resultados em /tmp/claude-1000/-home-ubuntu-meridianseal/e907fbb6-d370-4f89-bbaa-cae21594eb8d/scratchpad/t2run/resultados
